# Building an Eval Correction Loop: Teaching Your Evaluator What 'Good' Means for Your Domain

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/falcon-ai-page/evaluation/eval-correction-loop.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/falcon-ai-page/evaluation/eval-correction-loop.ipynb)

| Time | Difficulty |
|------|------------|
| 15 min | Intermediate |

By the end of this cookbook you will have a custom eval that catches failures a generic eval misses (off-policy refund offers, upsells in support replies, anything that violates your team's domain rules), 100% agreement with human verdicts on a four-row demo batch where the baseline scored 50%, and a repeatable loop to recalibrate after every prompt change. The only application-code change is one API call to register the new eval template.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- Python 3.9+

## Install

Install the FutureAGI evaluation SDK and set your API keys.

In [ ]:
%pip install ai-evaluation requests

In [ ]:
import os
os.environ["FI_API_KEY"] = "your_fi_api_key_here"
os.environ["FI_SECRET_KEY"] = "your_fi_secret_key_here"


## What is an eval correction loop?

Built-in evals like `is_helpful` or `tone` score the **surface form** of a response. They have no way to know your refund-escalation policy, your no-upsell rule, or your security team's blocklist. So a reply that looks polite, on-topic, and complete will pass a generic eval even when your team would flag it as bad.

The correction loop closes that gap in five steps: score a batch with a generic eval, mark the rows where the eval and your team disagree, encode those disagreements as few-shot FAIL examples in a custom rule prompt, re-score to verify the eval now matches your judgment, and iterate until fresh batches stay above your agreement bar. The example below runs the full loop on a four-row SaaS support batch where the bad replies look helpful but violate domain rules a generic eval cannot see.

## Step 1: Score the batch with a built-in eval

Start with a built-in template that scores the **surface form** of the response (`is_helpful`, `tone`, `coherence`). These are the evals most likely to pass replies your team would fail, because they have no way to know your domain rules. That's exactly the gap the correction loop closes. Run on a small batch (4 to 10 rows is enough for the first pass) and capture both the verdict AND the judge's reason. The reasons tell you whether the eval missed your domain rules or correctly applied them.

In [ ]:
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

# Real support replies from your agent. The bad ones look helpful but
# violate domain-specific rules (refund escalation, no upsells) that
# a generic helpfulness eval has no way to know about.
samples = [
    {
        "id": "r1",
        "user_query": "I forgot my password and can't log in.",
        "agent_response": "Click 'Forgot password' on the sign-in page and follow the email link. If the email doesn't arrive within 5 minutes, reply here with a screenshot and I'll resend it from our end.",
        "human_verdict": "good",
    },
    {
        "id": "r2",
        "user_query": "I want a refund for last month's subscription. The service was down 3 times.",
        "agent_response": "I'm sorry to hear about the disruption. I'll process a full refund for last month's $49 today, and you'll see it in your account within 7 to 10 business days.",
        "human_verdict": "bad",  # all refund offers must be escalated to a manager per policy
    },
    {
        "id": "r3",
        "user_query": "Why is my bill higher than expected this month?",
        "agent_response": "Your Pro plan has overage fees for usage above 10GB. You used 15GB this billing cycle, which added $12 to your bill. By the way, our Premium plan includes unlimited storage and the new collaboration features. Want me to upgrade you?",
        "human_verdict": "bad",  # answer is correct but the upsell is a brand-voice violation
    },
    {
        "id": "r4",
        "user_query": "How do I export my data?",
        "agent_response": "Go to Settings > Data > Export. You can pick CSV or JSON, and we'll email a download link within 30 minutes.",
        "human_verdict": "good",
    },
]

baseline_results = []
for s in samples:
    r = evaluator.evaluate(
        eval_templates="is_helpful",
        inputs={"input": s["user_query"], "output": s["agent_response"]},
        model_name="turing_flash",
    )
    baseline_results.append({
        "id": s["id"],
        "eval_score": r.eval_results[0].output,
        "eval_reason": r.eval_results[0].reason,
        "human_verdict": s["human_verdict"],
    })

for row in baseline_results:
    print(f"{row['id']}: eval={row['eval_score']!s:>5} | human={row['human_verdict']:>4} | {row['eval_reason'][:80]}")

The built-in `is_helpful` eval will likely return `Passed` for `r2` and `r3`. Both replies are on-topic, well-formed, and offer a concrete action. Nothing about the surface form gives the generic judge a reason to fail them. Your team flags them as bad because they violate domain rules the judge has no way to know about. That's the disagreement signal the correction loop will fix.

## Step 2: Identify the disagreements

The rows where the eval and human agree don't teach the eval anything new. The rows where they disagree are the entire point: they are the failure modes the generic eval can't see. Sort those out, then look at the judge's reason for each disagreement. The reason explains WHY the surface form passed the generic eval, which tells you exactly what your custom rule prompt needs to forbid.

In [ ]:
def passed(score):
    return str(score).strip().lower() == "passed"

disagreements = [
    r for r in baseline_results
    if passed(r["eval_score"]) != (r["human_verdict"] == "good")
]

print(f"{len(disagreements)} / {len(baseline_results)} disagreed with humans")
for r in disagreements:
    print(f"  {r['id']}: eval said {r['eval_score']}, human said {r['human_verdict']}")
    print(f"    reason: {r['eval_reason'][:120]}")

Pick 2 or 3 disagreement rows that capture distinct failure modes (here: out-of-policy refund commits, in-support upsells). Those become your few-shot examples in the next step.

## Step 3: Encode the corrections as a custom eval

A custom eval is just a rule prompt plus an output type. The rule prompt has two jobs. First, enumerate your domain rules in plain English so the judge model has criteria instead of vibes. Second, include 1 to 3 few-shot examples of FAILs your team has flagged so the judge knows what "FAIL" actually looks like in *your* domain. One API call to `/model-hub/create_custom_evals/` registers the template. Future eval calls reference it by name.

In [ ]:
import requests

rule_prompt = """\
You evaluate customer-support replies for a SaaS product.

A reply passes ONLY if ALL of the following hold:
1. Stays focused on the user's specific issue. No marketing language, no upsells, no pivots to other products.
2. Gives a concrete next step (a procedure, a link, a timeline, or a specific owner).
3. Does NOT commit to a refund, credit, or policy exception. Front-line agents must acknowledge the request and escalate to a manager.
4. Does NOT instruct the user to disable security features (2FA, MFA, encryption) as a workaround.

Examples of FAIL replies (learn from these):
- "I'm sorry to hear about the disruption. I'll process a full refund for last month's $49 today, and you'll see it in your account within 7 to 10 business days."
  -> FAIL: rule 3. Front-line agents can't commit to refunds. Should acknowledge and escalate.
- "Your Pro plan has overage fees for usage above 10GB. You used 15GB this billing cycle, which added $12 to your bill. By the way, our Premium plan includes unlimited storage and the new collaboration features. Want me to upgrade you?"
  -> FAIL: rule 1. Pivots from billing question to a sales pitch.

Example of a PASS reply:
- "Click 'Forgot password' on the sign-in page and follow the email link. If the email doesn't arrive within 5 minutes, reply here with a screenshot and I'll resend it from our end."
  -> PASS: focused on the issue, concrete next step, clear escalation path.

Now evaluate this reply.

User query: {{user_query}}
Agent response: {{agent_response}}
"""

resp = requests.post(
    "https://api.futureagi.com/model-hub/create_custom_evals/",
    headers={
        "X-Api-Key": os.environ["FI_API_KEY"],
        "X-Secret-Key": os.environ["FI_SECRET_KEY"],
    },
    json={
        "name": "support_reply_quality_v1",
        "description": "Domain-calibrated support-reply evaluator with policy and tone rules.",
        "criteria": rule_prompt,
        "output_type": "Pass/Fail",
        "required_keys": ["user_query", "agent_response"],
        "config": {"model": "turing_flash"},
    },
)
print(resp.json())
# {"status": True, "result": {"eval_template_id": "<uuid>"}}
# `status` is the API success flag; `result.eval_template_id` is the new template's
# UUID. The template is referenced by the `name` you passed ("support_reply_quality_v1")
# when you call `evaluator.evaluate(eval_templates=...)` in the next step.

Two things make this work. First, the rule prompt enumerates the domain rules explicitly, so the judge model has criteria instead of vibes. Second, the few-shot examples cover the exact failure modes you found in step 2, so the judge sees what "FAIL" looks like for *your* domain.

> **Tip.** Version your eval names (`_v1`, `_v2`). Each iteration creates a new template so historical eval runs stay reproducible. You can compare v1 vs v2 head-to-head later.

## Step 4: Re-score the same batch and measure agreement

Re-score the **same** batch with the new eval. Same samples, same human verdicts, only the evaluator changed, so any agreement delta is fully attributable to your rule prompt. Track the percentage of rows where eval and human agree. That's your calibration metric and the number you'll watch climb across iterations.

In [ ]:
calibrated_results = []
for s in samples:
    r = evaluator.evaluate(
        eval_templates="support_reply_quality_v1",
        inputs={"user_query": s["user_query"], "agent_response": s["agent_response"]},
    )
    calibrated_results.append({
        "id": s["id"],
        "eval_score": r.eval_results[0].output,
        "human_verdict": s["human_verdict"],
    })

agreement = sum(
    1 for r in calibrated_results
    if passed(r["eval_score"]) == (r["human_verdict"] == "good")
)
print(f"agreement: {agreement} / {len(samples)} ({100 * agreement / len(samples):.0f}%)")
for r in calibrated_results:
    match = "OK" if passed(r["eval_score"]) == (r["human_verdict"] == "good") else "MISS"
    print(f"  {match} {r['id']}: eval={r['eval_score']} human={r['human_verdict']}")

Expect a jump from around 50% baseline to 100% on this set. `r2` and `r3` now fail correctly because the rule prompt explicitly forbids out-of-policy refund commits and in-support upsells. `is_helpful` had no way to know either rule existed.

## Step 5: Iterate when agreement plateaus below your bar

One pass rarely catches every failure mode. New disagreements on fresh batches are the signal that your rule prompt missed a category. The loop continues with disciplined stop rules: don't add an example for an edge case the eval already gets right (it adds prompt length without changing behavior), and don't bloat past 8 to 10 examples (past that, agreement gains plateau and inference cost keeps growing).

If agreement is still below where you need it (typical bar: 85%+ on a held-out batch):

1. Pull a fresh sample of 20 to 30 rows the eval hasn't seen.
2. Re-score with the latest version (`support_reply_quality_v1`).
3. Find the new disagreements. These are failure modes your rule prompt didn't cover.
4. Rev to `_v2`: add 1 or 2 new few-shot examples or sharpen one of the rules.

In [ ]:
# After collecting fresh disagreements...
rule_prompt_v2 = rule_prompt + """

Additional FAIL example (learn from this):
- "Try disabling 2FA temporarily so you can log in, then re-enable it once you're past the issue."
  -> FAIL: rule 4. Never instruct users to disable security features. Offer a recovery code or escalate to security ops.
"""

# Re-register as support_reply_quality_v2 and compare scores side-by-side.

A well-calibrated eval typically converges in 2 or 3 iterations. Stop when fresh batches stay above your agreement bar.

## What you solved

Repetitive prompt iteration with no objective signal (the kind every team hits when "is this output good?" depends on policy rules a generic eval can't see) now becomes a measurable loop. You have a domain-calibrated evaluator, a single number (agreement %) to track across prompt changes, and a versioned trail of eval templates so you can compare quality runs head-to-head over time.

> **Check.** You ran a built-in eval, found rows where it disagreed with human judgment, encoded those corrections as a custom eval with explicit rules and few-shot failure examples, then re-scored to confirm the eval now matches how your team defines quality.

- **Generic evals scoring surface form** (instead of domain rules): solved by encoding rules in a custom rule prompt with few-shot FAIL examples
- **No signal which rows to focus on**: solved by surfacing only the disagreements as the calibration target
- **Starting from scratch on every prompt change**: solved by versioning the eval template (`_v1`, `_v2`) so old runs stay reproducible
- **No way to measure if the eval matches your team's bar**: solved by tracking agreement % between eval and human verdict across iterations

## Explore further

- **[Create Custom Evals](https://docs.futureagi.com/docs/evaluation/features/custom)**: Full reference for the custom eval template API
- **[Future AGI Models](https://docs.futureagi.com/docs/evaluation/features/futureagi-models)**: Pick the right judge model: turing_small, turing_flash, turing_large
- **[Eval Templates](https://docs.futureagi.com/docs/evaluation/concepts/eval-templates)**: Built-in vs custom templates and required-key conventions